# Lab 2 : Cosine similarity, the one math idea you need

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.

## What we are achieving in this lab

**Objective.** Measure how close two embeddings are. Lab 1 gave you the vectors. This lab gives you the ruler.

**Prerequisites.** Lab 1 finished. The same OpenAI key in `.env`. NumPy is already in `requirements.txt`.

**What this lab uses.**

| Layer | What it does | What we use | Why this one |
|-------|----------------|-------------|--------------|
| Embedding model | Turns text into a vector | OpenAI `text-embedding-3-small` | Same object as Lab 1. Do not mix models. |
| Library | Talks to OpenAI | LangChain `OpenAIEmbeddings` | Same interface as Lab 1. |
| Score | Says how close two vectors are | **Cosine similarity** (NumPy, three lines) | The default score for text embeddings. Vector databases use this (or the same ranking) under the hood. |

**What you will do.**

1. Reload the Lab 1 embeddings object.
2. Write cosine similarity and check that a vector vs itself is `1.0`.
3. Score real support sentences against one question and rank them.
4. See why teams use cosine, not "subtract the lists," and when a score is high enough to keep a chunk.

**What you should see.**

- Identical text scores **1.0**.
- A paraphrase of "reset my password" scores high.
- An unrelated question scores clearly lower.
- The **order** of the list is the lesson, not the third decimal.

**Cost.** A few more embedding calls. Fractions of a cent.

## Where this sits after Lab 1

Lab 1: an embedding is a list of 1536 numbers. Same length for a word or a paragraph. Similar meaning *tends* to look nearby if you stare at the first eight numbers.

Your eye cannot search 1536 slots. Production search does this instead:

1. Embed the question (`embed_query`).
2. Compare that vector to every stored chunk vector.
3. Keep the closest few chunks.
4. (Later labs) send those chunks to a chat model.

Step 2 is this lab. The name of that comparison is **cosine similarity**.

If you skip it, you can still print vectors. You cannot *search*.

## Ways to score two vectors (implement one, name the others)

You will write **cosine** only. Job conversations still mention the rest. Same pattern as Lab 1: one tool in the cells, a short map around it.

| Score | What it asks | When you will hear it |
|-------|----------------|------------------------|
| **Cosine similarity** | Do these two arrows point the same way? | Default for text embeddings. This lab. Range **-1 to 1**. |
| **Dot product** | Same as cosine **if** both arrows have length 1. | Pinecone / many APIs expose "dot" or "cosine." OpenAI v3 vectors are usually already length 1, so the ranking matches cosine. |
| **Euclidean distance (L2)** | How far apart are the two tips? | Cares about **length**, not only direction. A long paragraph can look "far" even if it is about the same topic. |
| **Manhattan distance (L1)** | Add up the gap in each slot. | Rare for text RAG. You may see it in older ML notes. |
| **Reranker (cross-encoder)** | Read the question and the chunk *together*, then score. | Second stage after cosine. Voyage `rerank-2.5`, Cohere rerank. Not a vector formula. Do not implement it today. |

**Grounded rule.** Pick cosine until you have a reason not to. Changing the score without re-checking retrieval quality is the same class of bug as mixing embedding models.

## The one picture

Treat each embedding as an **arrow** in a space with 1536 axes. You cannot draw that. The idea still holds in 2D:

- Two arrows pointing the same way → cosine near **1**. Same meaning (or the same sentence).
- Two arrows at a right angle → cosine near **0**. Unrelated.
- Two arrows pointing opposite ways → cosine near **-1**. Rare for these text models.

"Close" here does **not** mean "the lists look the same when printed." It means **the arrows point the same direction**.

The formula, in words:

> Multiply matching slots, add them up (that is the **dot product**). Then divide by the length of each arrow. Dividing by length is what stops a long paragraph from winning just because it is long.

You do not need to derive this. You do need to recognise the three numbers **1**, **0**, and **-1**, and to know that **higher cosine means closer meaning** for this kind of model.

### Step 1. Same key as Lab 1

Nothing new. If this cell fails, fix Lab 1's `.env` first.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Missing OPENAI_API_KEY. Copy .env.example to .env at the repository root, "
        "paste the key, and restart the kernel."
    )

print("OPENAI_API_KEY : set")
print("cwd            :", Path.cwd())

OPENAI_API_KEY : set
cwd            : h:\Project\llmops-course\week03


### Step 2. The Lab 1 embeddings object, plus the ruler

`embed` is Lab 1's `embed_query`, stored as a NumPy array so we can do arithmetic.

`cosine` is new. Read the comments. Then check the safety test: a vector compared with **itself** must be `1.0`.

In [2]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

# Same model as Lab 1. Mixing models here would make cosine meaningless.
EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def embed(text: str) -> np.ndarray:
    # Lab 1: one string in, one vector out.
    return np.asarray(embeddings.embed_query(text), dtype=float)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    # Dot product = how much the arrows point together.
    # np.linalg.norm = length of one arrow.
    # Divide by both lengths so long text does not win by size.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


hello = embed("hello")
print("model     :", EMBED_MODEL)
print("dimension :", hello.shape[0], "  (Lab 1: this must stay 1536)")
print("cosine of a vector with itself :", round(cosine(hello, hello), 3), "  (must be 1.0)")

model     : text-embedding-3-small
dimension : 1536   (Lab 1: this must stay 1536)
cosine of a vector with itself : 1.0   (must be 1.0)


If that last number is not `1.0`, stop. Either `cosine` is wrong, or the two vectors did not come from the same model.

That check is also an ops habit: a quick self-similarity test after you wire a new embedder.

### Step 3. Application: rank answers to a support question

This is what retrieval *is*, on a tiny list: embed the question once, score every candidate, put the highest score first.

Do not hunt for a magic number yet. Read the **order**.

In [3]:
# The user's question.
reference = "How do I reset my password?"
ref_vec = embed(reference)

# Possible snippets we might have stored. Mix of paraphrases and noise.
candidates = [
    "How do I reset my password?",
    "I forgot my password, can you help?",
    "How do I change my login credentials?",
    "What time does the office open?",
    "Show me the company holiday calendar.",
    "How do I cook pasta?",
]

print("Question:", reference)
print("Score 1.0 = same direction. Lower = less related.")
print()

# Score each candidate. Store (score, text) so we can sort later.
scored = []
for text in candidates:
    score = cosine(ref_vec, embed(text))
    scored.append((score, text))
    print(round(score, 3), " ", text)

print()
print("Highest first (this order is the retrieval ranking):")
scored.sort(reverse=True)
for score, text in scored:
    print(round(score, 3), " ", text)

Question: How do I reset my password?
Score 1.0 = same direction. Lower = less related.

1.0   How do I reset my password?
0.637   I forgot my password, can you help?
0.652   How do I change my login credentials?
0.142   What time does the office open?
0.144   Show me the company holiday calendar.
0.216   How do I cook pasta?

Highest first (this order is the retrieval ranking):
1.0   How do I reset my password?
0.652   How do I change my login credentials?
0.637   I forgot my password, can you help?
0.216   How do I cook pasta?
0.144   Show me the company holiday calendar.
0.142   What time does the office open?


You should see a pattern like this (your third decimal will differ):

- Identical sentence: **1.0**
- Paraphrase ("forgot my password"): still high
- Same job, different words ("credentials"): still clearly above the noise
- Office hours, holidays, pasta: a drop you can see without squinting

That drop is why embedding search works. The model was not given our list as a cheat sheet. It mapped each sentence into the same 1536-dimensional space, and password-reset clustered.

**Application: a threshold.** After ranking, a product still has to decide "is the top chunk good enough to show the model?" Teams often keep chunks above a floor (you will hear **0.7** or **0.75**). That number is **not a law**. It looked good on *someone else's* documents.

- If every chunk is about passwords, scores bunch together. You may need a higher floor, or a reranker.
- If the handbook is mixed, 0.75 might already throw away a useful paraphrase.

The grounded habit: measure the floor on **your** questions, write it down next to the embedding model id, and do not copy it from a blog. Lab 7 is where a naive floor hurts you.

### Step 4. Why cosine, not "subtract the two lists"

A common question: why not Euclidean distance (straight-line gap between the two tips)?

- **Euclidean** cares about **length**. A one-word query and a long paragraph can sit far apart even when they are about the same thing.
- **Cosine** cares about **direction**. Lab 1 already showed both texts become 1536 numbers. Cosine is how we compare those two arrows fairly.

Many embedding APIs (including OpenAI v3) return vectors whose length is already **1**. Then cosine and dot product give the **same ranking**. We still write the full cosine formula so the code stays correct if you later switch to a vendor that does not normalise.

The cell below is that check, not extra theory.

In [4]:
a = embed("reset password")
b = embed("forgot password")

dot = float(np.dot(a, b))
cos = cosine(a, b)

print("dot product       :", round(dot, 3))
print("cosine similarity :", round(cos, 3))
print("If these two match, both vectors already had length 1.")

dot product       : 0.735
cosine similarity : 0.736
If these two match, both vectors already had length 1.


## What you should be able to explain

> "Cosine similarity is a number from -1 to 1. For text embeddings, higher means the two pieces of text point the same way in vector space, which we treat as closer in meaning."

> "I use the same embedding model for the question and the documents. Mixing models makes cosine meaningless."

> "The default score is cosine. Dot product ranks the same when vectors have length 1. Euclidean cares about length, so I do not use it as my first text-RAG score. A reranker is a second stage, not a replacement for this lab."

> "A retrieve threshold is measured on my data. I log it next to the model id. I do not ship 0.75 because a tutorial did."

**Lab 3** is search: a question, a pile of snippets, keep the closest. Cosine is one way to score that match. The older way does not use vectors at all (keyword / word overlap). You will run both on the same question so you see what cosine is *for*, and why production RAG still keeps keywords next to it.